# Nachvollziehbare Prüfung: 525145–505377

Diese Auswertung liest die separat gespeicherten Nachweise. Sie verändert weder Projekt noch XTF. Screenshotwerte wurden visuell abgelesen; siehe VERGLEICH.md. Die vollständige C#-Simulation wurde vorher ausgeführt; Program.cs dokumentiert sie.


In [1]:
import json
from pathlib import Path
p = Path('nachweise')
vor = json.loads((p/'anzeige-vorher.json').read_text(encoding='utf-8-sig'))
nach = json.loads((p/'anzeige-nach-simulation.json').read_text(encoding='utf-8-sig'))
v = {r['Id']:r['Wert'] for r in vor}
print('Anzeigedefinitionen:', len(vor))
print('Gefüllt vorher:', sum(bool(r['Wert'].strip()) for r in vor))
print('Gefüllt nach Simulation:', sum(bool(r['Wert'].strip()) for r in nach))
for r in nach:
    if r['Wert'] != v[r['Id']]: print(r['Id'], repr(v[r['Id']]), '→', repr(r['Wert']))


Anzeigedefinitionen: 131
Gefüllt vorher: 32
Gefüllt nach Simulation: 37
haltung.profileref '' → 'ch24gwkd3obhLa8B'
haltung.operator '' → 'ch20p3q400002009'
haltung.tolevel '' → '505.910'
haltung.frompoint '' → 'ch24gwkdcFPcB0IM'
haltung.topoint '' → 'ch24gwkdsMxuOqnt'


In [2]:
import xml.etree.ElementTree as ET
import math
verbund = json.loads((p/'xtf-verbund.json').read_text(encoding='utf-8'))
xml = next(o['XML'] for o in verbund['Objekte'] if o['TID']=='ch24gwkdVy2uiPfJ')
ns = {'i':'http://www.interlis.ch/INTERLIS2.3'}
punkte = [(float(c.find('i:C1',ns).text),float(c.find('i:C2',ns).text)) for c in ET.fromstring(xml).findall('.//i:Verlauf//i:COORD',ns)]
anschluss_xml=json.loads((p/'einlauf-rueckverweise.json').read_text(encoding='utf-8'))[0]
c=ET.fromstring(anschluss_xml).find('.//i:Lage/i:COORD',ns)
q=(float(c.find('i:C1',ns).text),float(c.find('i:C2',ns).text))
a,b=punkte[:2]; d=tuple(y-x for x,y in zip(a,b))
t=sum((v-x)*di for v,x,di in zip(q,a,d))/sum(di*di for di in d)
assert 0 <= t <= 1
projection=tuple(x+t*di for x,di in zip(a,d))
print('Länge aus XTF-Geometrie:',sum(math.dist(a,b) for a,b in zip(punkte,punkte[1:])))
print('Anschlussdistanz ab Anfang:',t*math.dist(a,b))
print('Abstand Anschluss zur Linie:',math.dist(q,projection))
print('Gerundet:',round(t*math.dist(a,b),2))


Länge aus XTF-Geometrie: 11.440143356024418
Anschlussdistanz ab Anfang: 6.3573664879231035
Abstand Anschluss zur Linie: 0.0005825239218871568
Gerundet: 6.36


## Grenzen

Die Anschlussdistanz ist abgeleitet. Der Klartext „Einspitz“ wurde dadurch nicht nachgewiesen. Die alte Projektdatei bildet nicht zwingend ungespeicherte Eingaben ab. Die laufende Anwendung bestätigt Z4; alle übrigen Felder sind in VERGLEICH.md nach Quelle getrennt.
